# 22 · One tool call, end to end
Level L2 · Part 3 · core · ~35 min · builds on 20

## Goal
Declare Scout's `search_notes` tool, receive the model's request to call it, run it yourself, send the
result back with the matching id, and get a final answer: first with a fake model, then with Gemini.

## The idea
A model cannot run code. It can only *ask*. You send it a **tool schema**: a name, a description and a
JSON Schema for the arguments (the same JSON Schema as lesson 20). If the model decides it needs the
tool, its reply is not text but a **tool call**: "run `search_notes` with `query="Mars"`", tagged with
a **tool call id**. Your code runs the function and sends a **tool result** back, tagged with the same
id. A second model call reads that result and writes the answer. Think of a restaurant ticket: the
waiter writes order #17, the kitchen cooks it, and the plate goes out marked #17 so it reaches the
right table.

## Picture
```mermaid
sequenceDiagram
    participant S as Scout (your code)
    participant M as Model
    S->>M: question + tool schema
    M-->>S: tool call (id=call_1, search_notes, query="Mars")
    Note over S: run search_notes("Mars")
    S->>M: history + tool result (id=call_1)
    M-->>S: final answer (text)
```

## Step by step
### 1. Start the recorder and set up
Same as lessons 19 to 21: the recorder first. Without a key, the Gemini calls replay from
`../data/llm_cassettes/22_tool-calling-round-trip.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("22_tool-calling-round-trip")

LLM mode: replay (22_tool-calling-round-trip.yaml)


Setup from lessons 01 and 07: the notes, the client and the pinned model, plus `search_notes`, Scout's
first tool from lesson 01, unchanged.

In [2]:
import json
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

TOOLS = {"search_notes": search_notes}
print("model:", MODEL, "| tools:", list(TOOLS))

model: gemini-3.5-flash-lite | tools: ['search_notes']


### 2. The tool schema
The model never sees your Python function. It sees only this description of it. The `description`
fields matter: they are how the model decides *when* to call the tool and *what* to pass.

In [3]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}. Use it before answering any question "
                   "about what the notes say.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}},
        "required": ["query"],
    },
}
print(json.dumps(SEARCH_NOTES_SCHEMA["parameters"]))

{"type": "object", "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}}, "required": ["query"]}


### 3. A fake model that speaks in tool calls
Lesson 01's fake model returned `{"type": "tool_call", ...}`. Real APIs add two things: the tool call
sits inside an **assistant message** that goes into the history, and it carries an id. Our fake does the
same. It also enforces one rule: a tool result must answer a call id that it issued.

In [4]:
class FakeToolModel:
    def __init__(self):
        self.issued = set()                     # tool call ids handed out so far
        self.seen = []                          # every message list we were sent

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        self.seen.append([dict(m) for m in messages])
        last = messages[-1]
        if last["role"] == "user":              # a question: ask for the tool
            call_id = f"call_{len(self.issued) + 1}"
            self.issued.add(call_id)
            word = last["text"].rstrip("?").split()[-1]
            return {"role": "assistant", "tool_calls": [
                {"id": call_id, "name": tools[0]["name"], "args": {"query": word}}]}
        if last["role"] == "tool":              # a result: check the id, then answer from it
            if last["call_id"] not in self.issued:
                raise ValueError(f"tool result for unknown call id {last['call_id']!r}")
            ids = [n["id"] for n in json.loads(last["content"])]
            return {"role": "assistant", "text": f"Notes {ids} mention it."}
        raise ValueError(f"unexpected role {last['role']!r}")

### 4. The round trip with the fake model
**Call 1.** Send the question and the tool schema. The reply has no text, only a tool call.

In [5]:
fake = FakeToolModel()
QUESTION = "Which of my notes mention Mars?"
messages = [{"role": "user", "text": QUESTION}]
reply = fake(messages, tools=[SEARCH_NOTES_SCHEMA])
print(reply)

{'role': 'assistant', 'tool_calls': [{'id': 'call_1', 'name': 'search_notes', 'args': {'query': 'Mars'}}]}


**Run it.** The model only named a tool and gave arguments. We look the name up in `TOOLS` and call the
function. This is the only place code actually runs.

In [6]:
call = reply["tool_calls"][0]
result = TOOLS[call["name"]](**call["args"])
print(f"{call['name']}({call['args']}) -> {len(result)} notes:", [n["id"] for n in result])

search_notes({'query': 'Mars'}) -> 3 notes: [1, 3, 5]


**Call 2.** Append the model's own tool-call message, then the tool result with the same id. The result
is sent as text (JSON), because that is what goes over the wire.

In [7]:
messages.append(reply)
messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                 "content": json.dumps(result)})
final = fake(messages, tools=[SEARCH_NOTES_SCHEMA])
print(final["text"])
print("history roles:", [m["role"] for m in messages])

Notes [1, 3, 5] mention it.
history roles: ['user', 'assistant', 'tool']


Call 2 sent three messages: user, assistant (tool call), tool (result). The model needs the whole
history every time, because the API is stateless (lesson 08).

### 5. The same round trip with Gemini: declare the tool
Gemini calls a tool schema a `FunctionDeclaration`. Our JSON Schema goes in `parameters_json_schema`.
We keep automatic function calling off (`NO_AFC`), so the SDK never runs a tool behind our back.

In [8]:
search_decl = types.FunctionDeclaration(
    name=SEARCH_NOTES_SCHEMA["name"],
    description=SEARCH_NOTES_SCHEMA["description"],
    parameters_json_schema=SEARCH_NOTES_SCHEMA["parameters"],
)
TOOL_CFG = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[search_decl])],
                                       automatic_function_calling=NO_AFC, max_output_tokens=300)
contents = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]
print(search_decl.name, "| params:", list(search_decl.parameters_json_schema["properties"]))

search_notes | params: ['query']


**Call 1.** Same question, now to the real model.

In [9]:
resp1 = client.models.generate_content(model=MODEL, contents=contents, config=TOOL_CFG)
print("finish_reason:", resp1.candidates[0].finish_reason)
print("text parts:", [p.text for p in resp1.candidates[0].content.parts if p.text])
for fc in resp1.function_calls:
    print("tool call:", fc.id, fc.name, fc.args)

finish_reason: FinishReason.STOP
text parts: []
tool call: call_131434 search_notes {'query': 'Mars'}


Gemini chose the tool and picked `query` itself. Note `finish_reason` is `STOP`: unlike Anthropic's
`tool_use` stop reason, Gemini tells you about a tool call only through `function_calls`, so check that
list. The id is Gemini's **tool call id**, not lesson 13's call id (which Scout makes for logging).

Look at the whole reply part: besides the call, Gemini 3 adds a `thought_signature`.

In [10]:
part = resp1.candidates[0].content.parts[0]
print("role:", resp1.candidates[0].content.role)
print("part fields:", sorted(part.model_dump(exclude_none=True)))

role: model
part fields: ['function_call', 'thought_signature']


**Run it.** Exactly as with the fake: look up the name, call the function with the model's arguments.

In [11]:
fc = resp1.function_calls[0]
hits = TOOLS[fc.name](**fc.args)
print(f"{fc.name}({dict(fc.args)}) ->", [n["id"] for n in hits])

search_notes({'query': 'Mars'}) -> [1, 3, 5]


**Call 2.** Append the model's turn *unchanged* (it carries the thought signature), then the result as a
`function_response` part with the same id and name. Gemini has no `tool` role: the result goes in a
`user` turn. `response` must be a dict, so the list goes under a `"result"` key.

In [12]:
contents.append(resp1.candidates[0].content)
contents.append(types.Content(role="user", parts=[types.Part(function_response=types.FunctionResponse(
    id=fc.id, name=fc.name, response={"result": hits}))]))
resp2 = client.models.generate_content(model=MODEL, contents=contents, config=TOOL_CFG)
print("finish_reason:", resp2.candidates[0].finish_reason, "| tool calls:", resp2.function_calls)
print(resp2.text)

finish_reason: FinishReason.STOP | tool calls: None
Your notes that mention Mars are:

1. **Note 1:** "Mars has two small moons, Phobos and Deimos."
2. **Note 3:** "A day on Mars lasts about 24 hours and 39 minutes."
3. **Note 5:** "The rover Perseverance landed on Mars in February 2021."


A text answer built from the notes our code found. Two model calls, one tool run.

### 6. Fake format and Gemini side by side
The shapes differ, the round trip does not. Lesson 23 adds the Anthropic and OpenAI wire formats.

| Concept | Our fake | Gemini `generate_content` |
|---|---|---|
| tool schema | `{"name", "description", "parameters"}` | `FunctionDeclaration(..., parameters_json_schema=...)` |
| tool call | `reply["tool_calls"]`, role `assistant` | `resp.function_calls`, part `function_call`, role `model` |
| tool result | role `tool`, `call_id`, `content` | part `function_response` (`id`, `name`, `response`), role `user` |
| "a tool was called" | `tool_calls` present | `function_calls` not empty (finish reason stays `STOP`) |

The final `contents` list holds the four turns we sent and received:

In [13]:
for c in contents + [resp2.candidates[0].content]:
    kinds = [k for p in c.parts for k in ("text", "function_call", "function_response") if getattr(p, k)]
    print(f"{c.role:5} {kinds}")

user  ['text']
model ['function_call']
user  ['function_response']
model ['text']


## At work
- **Tool definitions cost tokens on every call.** The schemas are sent as input each time (Anthropic adds
  a ~286-token tool system prompt on top for Sonnet 5.5 / Opus 5.5), so teams keep descriptions tight
  and the tool list short. → lesson 29 (designing good tools), lesson 54 (cost controls)
- **Never trust the arguments.** A schema shapes `args`, but teams still validate them with Pydantic
  before running anything, and return failures to the model instead of crashing. → lesson 24 (strict
  schemas), lesson 28 (tool errors the model can recover from)
- **The tool call id is what keeps results straight** once a model asks for several tools in one turn;
  every result must carry its own id. → lesson 25 (parallel tool calls)
- **Tool results are untrusted input.** A note could contain "ignore your instructions". Production code
  keeps results inside tool-result parts, never pasted into the system prompt. → lesson 51

## What just happened
- We described `search_notes` with a **tool schema**: name, description, JSON Schema arguments.
- The model replied with a **tool call** (name, args, id) and no text. It ran nothing.
- Our code looked the name up in `TOOLS` and ran the function.
- We sent back the model's turn plus a **tool result** carrying the same **tool call id**, and a second
  call produced the answer.
- On Gemini: `FunctionDeclaration` in, `resp.function_calls` out, `function_response` back in a `user`
  turn, and the model's turn re-sent unchanged because of its thought signature.

## Common mistakes
**1. Answering with the wrong id.** Copying an id by hand, or reusing one from an earlier turn, sends a
result the model never asked for. Our fake rejects it, so the bug shows up at once instead of as a
confused answer.

In [14]:
bad_fake = FakeToolModel()
msgs = [{"role": "user", "text": QUESTION}]
msgs.append(bad_fake(msgs, tools=[SEARCH_NOTES_SCHEMA]))
msgs.append({"role": "tool", "call_id": "call_0", "name": "search_notes", "content": json.dumps(result)})
bad_fake(msgs, tools=[SEARCH_NOTES_SCHEMA])

ValueError: tool result for unknown call id 'call_0'

Fix: take the id from the tool call you are answering, never type it.

In [15]:
msgs[-1]["call_id"] = msgs[-2]["tool_calls"][0]["id"]
print(bad_fake(msgs, tools=[SEARCH_NOTES_SCHEMA])["text"])

Notes [1, 3, 5] mention it.


**2. Rebuilding the model's turn yourself.** It looks harmless to re-create the model turn from just the
function call. That drops the thought signature, and Gemini 3 rejects the request with a 400.

In [16]:
rebuilt = types.Content(role="model", parts=[types.Part(function_call=fc)])     # no thought_signature
client.models.generate_content(model=MODEL, contents=[contents[0], rebuilt, contents[2]], config=TOOL_CFG)

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Function call is missing a thought_signature in functionCall parts. This is required for tools to work correctly, and missing thought_signature may lead to degraded model performance. Additional data, function call `default_api:search_notes` , position 2. Please refer to https://ai.google.dev/gemini-api/docs/thought-signatures for more details.', 'status': 'INVALID_ARGUMENT'}}

Fix: append `resp.candidates[0].content` as it came back, as step 5 did. That is the `contents` list
that already produced the answer in call 2.

In [17]:
print([type(p.thought_signature).__name__ for p in contents[1].parts])

['bytes']


## Try it
Add a second tool, `count_notes()` (no arguments, returns `{"count": len(NOTES)}`). Write its schema,
put both schemas in the Gemini config, ask "How many notes do I have?", run whichever tool Gemini calls
with the matching id, and print the final answer.

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **tool schema** — the description of a tool sent to the model: name, description and a JSON Schema for its arguments (Gemini: `FunctionDeclaration`).
- **tool call** — a model reply that asks your code to run a named tool with given arguments; the model runs nothing itself.
- **tool result** — the message your code sends back with the tool's output, tagged with the tool call id it answers.
- **tool call id** — the id the model gives each tool call; the matching tool result must carry the same id (not the same as lesson 13's call id).
- **thought signature** — an opaque field Gemini 3 attaches to function-call parts; send the model's turn back unchanged or the request fails.